In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [ ]:
# Generate DimDate - Calendar dimension for 18 months (gives us context before/after our 12-month job period)
def generate_dim_date(start_date, end_date):
    """Generate a date dimension table"""
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
    dim_date = pd.DataFrame({
        'DateKey': dates.strftime('%Y%m%d').astype(int),  # e.g., 20250101
        'Date': dates,
        'Year': dates.year,
        'Quarter': dates.quarter,
        'Month': dates.month,
        'MonthName': dates.strftime('%B'),
        'Week': dates.isocalendar().week,
        'DayOfWeek': dates.dayofweek + 1,  # 1=Monday, 7=Sunday
        'DayName': dates.strftime('%A'),
        'IsWeekend': (dates.dayofweek >= 5).astype(int)
    })
    
    return dim_date

# Generate 18 months of dates (Jan 2025 - June 2026)
start_date = '2025-01-01'
end_date = '2026-06-30'

dim_date = generate_dim_date(start_date, end_date)

# Preview the data
print(f"Generated {len(dim_date)} dates")
dim_date.head(10)

In [ ]:
# Generate DimRegion - Geographic regions/branches
def generate_dim_region():
    """Generate region/branch dimension"""
    regions = [
        {'RegionID': 1, 'RegionName': 'Calgary', 'Province': 'Alberta', 'RegionType': 'Metro'},
        {'RegionID': 2, 'RegionName': 'Edmonton', 'Province': 'Alberta', 'RegionType': 'Metro'},
        {'RegionID': 3, 'RegionName': 'Red Deer', 'Province': 'Alberta', 'RegionType': 'Regional'},
        {'RegionID': 4, 'RegionName': 'Lethbridge', 'Province': 'Alberta', 'RegionType': 'Regional'},
        {'RegionID': 5, 'RegionName': 'Grande Prairie', 'Province': 'Alberta', 'RegionType': 'Regional'}
    ]
    
    return pd.DataFrame(regions)

dim_region = generate_dim_region()

print(f"Generated {len(dim_region)} regions")
dim_region

In [ ]:
# Generate DimClient - Customer/client dimension
def generate_dim_client(num_clients=20):
    """Generate client dimension with different segments"""
    
    client_segments = ['Residential', 'Commercial', 'Industrial', 'Government']
    client_tiers = ['A', 'B', 'C']  # A = high-value, C = lower-value
    
    clients = []
    for i in range(1, num_clients + 1):
        segment = random.choice(client_segments)
        
        # Tier distribution: more B and C tier clients than A tier (realistic)
        tier = random.choices(client_tiers, weights=[15, 40, 45])[0]
        
        clients.append({
            'ClientID': i,
            'ClientName': f'Client_{i:03d}',
            'ClientSegment': segment,
            'ClientTier': tier,
            'YearsAsClient': random.randint(1, 10)
        })
    
    return pd.DataFrame(clients)

dim_client = generate_dim_client(num_clients=20)

print(f"Generated {len(dim_client)} clients")
dim_client.head(10)

In [ ]:
# Generate DimPM - Project Manager dimension
def generate_dim_pm(num_pms=8):
    """Generate project manager dimension"""
    
    pm_names = ['Sarah Chen', 'Mike O\'Brien', 'Jessica Martinez', 'David Kim', 
                'Emily Thompson', 'James Wilson', 'Maria Garcia', 'Robert Taylor',
                'Amanda Lee', 'Chris Anderson']
    
    pms = []
    for i in range(1, num_pms + 1):
        years_exp = random.randint(2, 15)
        
        # Experience correlates with performance tier
        if years_exp >= 10:
            tier = 'Senior'
        elif years_exp >= 5:
            tier = 'Mid-Level'
        else:
            tier = 'Junior'
        
        pms.append({
            'PMID': i,
            'PMName': pm_names[i-1] if i <= len(pm_names) else f'PM_{i}',
            'YearsExperience': years_exp,
            'PMTier': tier,
            'Region': random.choice(['Calgary', 'Edmonton', 'Red Deer', 'Lethbridge', 'Grande Prairie'])
        })
    
    return pd.DataFrame(pms)

dim_pm = generate_dim_pm(num_pms=8)

print(f"Generated {len(dim_pm)} project managers")
dim_pm

In [ ]:
# Generate FactJobs - Main job/project fact table
def generate_fact_jobs(num_jobs=50, dim_client=None, dim_pm=None, dim_region=None):
    """Generate job fact table with realistic distributions"""
    
    job_types = ['New Construction', 'Renovation', 'Maintenance', 'Emergency Repair']
    job_statuses = ['Completed', 'In Progress', 'Delayed', 'On Hold']
    
    jobs = []
    
    # Job start dates spread across 12 months (Feb 2025 - Jan 2026)
    start_date = datetime(2025, 2, 1)
    end_date = datetime(2026, 1, 31)
    
    for job_id in range(1, num_jobs + 1):
        # Random start date within our range
        days_offset = random.randint(0, (end_date - start_date).days)
        job_start = start_date + timedelta(days=days_offset)
        
        # Job duration: 5-90 days depending on type
        job_type = random.choice(job_types)
        if job_type == 'Emergency Repair':
            base_duration = random.randint(1, 10)
        elif job_type == 'Maintenance':
            base_duration = random.randint(5, 30)
        elif job_type == 'Renovation':
            base_duration = random.randint(15, 60)
        else:  # New Construction
            base_duration = random.randint(30, 90)
        
        # Estimated completion
        estimated_end = job_start + timedelta(days=base_duration)
        
        # Actual completion (some jobs are delayed, some on time, some early)
        delay_factor = random.choices(
            [-5, 0, 5, 10, 20],  # days early/on-time/delayed
            weights=[10, 50, 20, 15, 5]  # most on-time, some delayed
        )[0]
        
        actual_duration = base_duration + delay_factor
        actual_end = job_start + timedelta(days=actual_duration)
        
        # Status based on dates
        today = datetime(2026, 1, 29)  # Current date
        if actual_end < today:
            status = 'Completed'
        elif delay_factor > 10:
            status = 'Delayed'
        elif random.random() < 0.05:  # 5% on hold
            status = 'On Hold'
        else:
            status = 'In Progress'
        
        # Contract value and costs
        contract_value = random.randint(10000, 500000)
        
        # Estimated costs (should be less than contract for profit)
        estimated_cost = contract_value * random.uniform(0.65, 0.85)
        
        # Actual costs vary (sometimes over budget)
        cost_variance = random.choices(
            [0.95, 1.0, 1.05, 1.15],  # under, on, slightly over, way over budget
            weights=[20, 50, 20, 10]
        )[0]
        actual_cost = estimated_cost * cost_variance
        
        # Estimated hours
        estimated_hours = int(estimated_cost / 75)  # ~$75/hour average
        actual_hours = int(actual_cost / 75)
        
        jobs.append({
            'JobID': job_id,
            'ClientID': random.choice(dim_client['ClientID'].values),
            'PMID': random.choice(dim_pm['PMID'].values),
            'RegionID': random.choice(dim_region['RegionID'].values),
            'JobType': job_type,
            'JobStatus': status,
            'StartDate': job_start.strftime('%Y-%m-%d'),
            'EstimatedEndDate': estimated_end.strftime('%Y-%m-%d'),
            'ActualEndDate': actual_end.strftime('%Y-%m-%d') if status == 'Completed' else None,
            'ContractValue': round(contract_value, 2),
            'EstimatedCost': round(estimated_cost, 2),
            'ActualCost': round(actual_cost, 2) if status == 'Completed' else round(actual_cost * random.uniform(0.3, 0.7), 2),
            'EstimatedHours': estimated_hours,
            'ActualHours': actual_hours if status == 'Completed' else int(actual_hours * random.uniform(0.3, 0.7))
        })
    
    return pd.DataFrame(jobs)

fact_jobs = generate_fact_jobs(num_jobs=50, dim_client=dim_client, dim_pm=dim_pm, dim_region=dim_region)

print(f"Generated {len(fact_jobs)} jobs")
print(f"Job status breakdown:")
print(fact_jobs['JobStatus'].value_counts())
print(f"\nJob type breakdown:")
print(fact_jobs['JobType'].value_counts())
fact_jobs.head(10)

In [9]:
# Generate FactJobPhaseEvents - Track phases within each job
def generate_fact_job_phase_events(fact_jobs):
    """Generate phase-level tracking for jobs"""
    
    # Standard phases for construction jobs
    phases = ['Planning', 'Mobilization', 'Execution', 'Inspection', 'Closeout']
    
    phase_events = []
    event_id = 1
    
    for _, job in fact_jobs.iterrows():
        job_start = datetime.strptime(job['StartDate'], '%Y-%m-%d')
        
        # Fix: Handle None/NaN values properly
        if pd.notna(job['ActualEndDate']):
            job_end_str = job['ActualEndDate']
        else:
            job_end_str = job['EstimatedEndDate']
            
        job_end = datetime.strptime(job_end_str, '%Y-%m-%d')
        
        total_duration = (job_end - job_start).days
        
        # Distribute duration across phases (rough percentages)
        phase_percentages = [0.10, 0.10, 0.60, 0.10, 0.10]  # Execution takes most time
        
        current_date = job_start
        
        for i, phase in enumerate(phases):
            phase_duration = int(total_duration * phase_percentages[i])
            if phase_duration < 1:
                phase_duration = 1
            
            planned_end = current_date + timedelta(days=phase_duration)
            
            # Add some variance to actual dates (some phases delayed, some on time)
            phase_delay = random.choices(
                [0, 2, 5, 10],
                weights=[60, 20, 15, 5]
            )[0]
            
            actual_end = planned_end + timedelta(days=phase_delay)
            
            # Phase status
            today = datetime(2026, 1, 29)
            if actual_end < today:
                phase_status = 'Completed'
            elif current_date <= today < actual_end:
                phase_status = 'In Progress'
            else:
                phase_status = 'Not Started'
            
            phase_events.append({
                'EventID': event_id,
                'JobID': job['JobID'],
                'Phase': phase,
                'PhaseSequence': i + 1,
                'PlannedStartDate': current_date.strftime('%Y-%m-%d'),
                'PlannedEndDate': planned_end.strftime('%Y-%m-%d'),
                'ActualStartDate': current_date.strftime('%Y-%m-%d') if phase_status != 'Not Started' else None,
                'ActualEndDate': actual_end.strftime('%Y-%m-%d') if phase_status == 'Completed' else None,
                'PhaseStatus': phase_status,
                'DaysDelayed': phase_delay if phase_status == 'Completed' else 0
            })
            
            event_id += 1
            current_date = actual_end
    
    return pd.DataFrame(phase_events)

fact_job_phase_events = generate_fact_job_phase_events(fact_jobs)

print(f"Generated {len(fact_job_phase_events)} phase events")
print(f"Phase status breakdown:")
print(fact_job_phase_events['PhaseStatus'].value_counts())
print(f"\nPhase breakdown:")
print(fact_job_phase_events['Phase'].value_counts())
fact_job_phase_events.head(15)

Generated 250 phase events
Phase status breakdown:
PhaseStatus
Completed      231
Not Started     15
In Progress      4
Name: count, dtype: int64

Phase breakdown:
Phase
Planning        50
Mobilization    50
Execution       50
Inspection      50
Closeout        50
Name: count, dtype: int64


,EventID,JobID,Phase,PhaseSequence,PlannedStartDate,PlannedEndDate,ActualStartDate,ActualEndDate,PhaseStatus,DaysDelayed
0,1,1,Planning,1,2026-01-14,2026-01-17,2026-01-14,2026-01-17,Completed,0
1,2,1,Mobilization,2,2026-01-17,2026-01-20,2026-01-17,2026-01-20,Completed,0
2,3,1,Execution,3,2026-01-20,2026-02-11,2026-01-20,NaN,In Progress,0
3,4,1,Inspection,4,2026-02-16,2026-02-19,NaN,NaN,Not Started,0
4,5,1,Closeout,5,2026-02-19,2026-02-22,NaN,NaN,Not Started,0
5,6,2,Planning,1,2025-09-28,2025-10-01,2025-09-28,2025-10-06,Completed,5
6,7,2,Mobilization,2,2025-10-06,2025-10-09,2025-10-06,2025-10-09,Completed,0
7,8,2,Execution,3,2025-10-09,2025-10-27,2025-10-09,2025-10-27,Completed,0
8,9,2,Inspection,4,2025-10-27,2025-10-30,2025-10-27,2025-11-04,Completed,5
9,10,2,Closeout,5,2025-11-04,2025-11-07,2025-11-04,2025-11-07,Completed,0


In [10]:
# Create a data folder and save all tables to CSV
import os

# Create data folder if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

# Save all dimension tables
dim_date.to_csv('data/DimDate.csv', index=False)
dim_region.to_csv('data/DimRegion.csv', index=False)
dim_client.to_csv('data/DimClient.csv', index=False)
dim_pm.to_csv('data/DimPM.csv', index=False)

# Save fact tables
fact_jobs.to_csv('data/FactJobs.csv', index=False)
fact_job_phase_events.to_csv('data/FactJobPhaseEvents.csv', index=False)

print("✅ All CSV files saved to 'data' folder!")
print("\nFiles created:")
print("  - DimDate.csv")
print("  - DimRegion.csv")
print("  - DimClient.csv")
print("  - DimPM.csv")
print("  - FactJobs.csv")
print("  - FactJobPhaseEvents.csv")

✅ All CSV files saved to 'data' folder!

Files created:
  - DimDate.csv
  - DimRegion.csv
  - DimClient.csv
  - DimPM.csv
  - FactJobs.csv
  - FactJobPhaseEvents.csv
